In [1]:
import pandas as pd
import numpy as np

# 读取数据
df = pd.read_csv('E_commerce.csv')

# 基本信息
print("数据形状:", df.shape)
print("\n数据类型:")
print(df.dtypes)
print("\n缺失值统计:")
print(df.isnull().sum())
print("\n前几行数据:")
print(df.head())

数据形状: (51289, 27)

数据类型:
Customer ID             object
Customer Name           object
Segment                 object
City                    object
State                   object
Country                 object
Region                  object
Gender                  object
Age                      int64
Education               object
Marital Status          object
Order ID                object
Order Date              object
Months                  object
Ship Mode               object
Product Category        object
Product                 object
Sales                   object
Quantity                object
Discount                object
Profit                  object
Shipping Cost           object
Order Priority          object
Browsing Time (min)    float64
Like                     int64
Share                    int64
Add to Cart              int64
dtype: object

缺失值统计:
Customer ID             0
Customer Name           0
Segment                 0
City                    0
State       

C:\Users\X\AppData\Local\Temp\ipykernel_31260\2933578920.py:5: DtypeWarning: Columns (18,19) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('E_commerce.csv')


In [3]:
# 清洗金额列：移除$符号和逗号，转换为浮点数
def clean_currency(x):
    if isinstance(x, str):
        return float(x.replace('$', '').replace(',', '').strip())
    return float(x)

# 应用清洗函数
amount_columns = ['Sales', 'Shipping Cost']
for col in amount_columns:
    for col in amount_columns:
        s = df[col].astype(str).str.strip()
        # handle parentheses negative format e.g. (1,234.56) -> -1234.56
        s = s.str.replace(r'^\((.*)\)$', r'-\1', regex=True)
        # remove $ and commas
        s = s.str.replace(r'[\$,]', '', regex=True)
        # empty strings -> NaN, coerce non-numeric to NaN
        df[col] = pd.to_numeric(s.replace('', np.nan), errors='coerce')

# 清洗Profit列 - 有些可能有空值或特殊字符
def clean_profit(x):
    try:
        if pd.isna(x):
            return np.nan
        if isinstance(x, str):
            return float(x.replace('$', '').replace(',', '').strip())
        return float(x)
    except:
        return np.nan

df['Profit'] = df['Profit'].apply(clean_profit)

# 清洗Discount列
def clean_discount(x):
    try:
        if pd.isna(x) or x in ['xxx', 'abc', 'test']:
            return np.nan
        if isinstance(x, str):
            # 如果已经是百分比格式，如'0.05'
            if '.' in x:
                return float(x)
            # 如果是整数，如'5'
            return float(x)
        return float(x)
    except:
        return np.nan

df['Discount'] = df['Discount'].apply(clean_discount)

# 清洗Quantity列
def clean_quantity(x):
    try:
        if pd.isna(x) or x in ['abc', '']:
            return np.nan
        return int(float(x))
    except:
        return np.nan

df['Quantity'] = df['Quantity'].apply(clean_quantity)

# 转换日期列
df['Order Date'] = pd.to_datetime(df['Order Date'], errors='coerce')

In [4]:
# 查看缺失值情况
missing_data = df.isnull().sum()
print("清洗前缺失值:")
print(missing_data[missing_data > 0])

# 填充缺失值策略
# Quantity: 用中位数填充
quantity_median = df['Quantity'].median()
df['Quantity'] = df['Quantity'].fillna(quantity_median)

# Discount: 用0填充（假设没有折扣）
df['Discount'] = df['Discount'].fillna(0)

# Profit: 用中位数填充
profit_median = df['Profit'].median()
df['Profit'] = df['Profit'].fillna(profit_median)

# Order Date: 如果有缺失，用最频繁的日期填充
mode_date = df['Order Date'].mode()[0]
df['Order Date'] = df['Order Date'].fillna(mode_date)

# Browsing Time: 用中位数填充
browsing_median = df['Browsing Time (min)'].median()
df['Browsing Time (min)'] = df['Browsing Time (min)'].fillna(browsing_median)

# Like, Share, Add to Cart: 用0填充（假设没有互动）
binary_columns = ['Like', 'Share', 'Add to Cart']
for col in binary_columns:
    df[col] = df[col].fillna(0).astype(int)

# Order Priority: 用'Medium'填充
df['Order Priority'] = df['Order Priority'].fillna('Medium')

清洗前缺失值:
Region            74
Sales              1
Quantity           2
Discount           1
Shipping Cost      1
Order Priority     2
dtype: int64


In [6]:
# 查看缺失值情况
missing_data = df.isnull().sum()
print("清洗前缺失值:")
print(missing_data[missing_data > 0])

# 填充缺失值策略
# Quantity: 用中位数填充
quantity_median = df['Quantity'].median()
df['Quantity'] = df['Quantity'].fillna(quantity_median)

# Discount: 用0填充（假设没有折扣）
df['Discount'] = df['Discount'].fillna(0)

# Profit: 用中位数填充
profit_median = df['Profit'].median()
df['Profit'] = df['Profit'].fillna(profit_median)

# Order Date: 如果有缺失，用最频繁的日期填充
mode_date = df['Order Date'].mode()[0]
df['Order Date'] = df['Order Date'].fillna(mode_date)

# Browsing Time: 用中位数填充
browsing_median = df['Browsing Time (min)'].median()
df['Browsing Time (min)'] = df['Browsing Time (min)'].fillna(browsing_median)

# Like, Share, Add to Cart: 用0填充（假设没有互动）
binary_columns = ['Like', 'Share', 'Add to Cart']
for col in binary_columns:
    df[col] = df[col].fillna(0).astype(int)

# Order Priority: 用'Medium'填充
df['Order Priority'] = df['Order Priority'].fillna('Medium')

清洗前缺失值:
Region           74
Sales             1
Shipping Cost     1
dtype: int64


In [5]:
# 验证数值范围
def validate_ranges():
    print("数值范围验证:")
    
    # Quantity应该在合理范围内
    print(f"Quantity范围: {df['Quantity'].min()} - {df['Quantity'].max()}")
    
    # Discount应该在0-1之间
    print(f"Discount范围: {df['Discount'].min()} - {df['Discount'].max()}")
    
    # Sales和Profit应该为正数或负数（利润可能为负）
    print(f"Sales范围: {df['Sales'].min()} - {df['Sales'].max()}")
    print(f"Profit范围: {df['Profit'].min()} - {df['Profit'].max()}")
    
    # Age应该在合理范围内（假设18-100）
    print(f"Age范围: {df['Age'].min()} - {df['Age'].max()}")
    
    # Browsing Time应该在合理范围内
    print(f"Browsing Time范围: {df['Browsing Time (min)'].min()} - {df['Browsing Time (min)'].max()}")

validate_ranges()

# 处理异常值
# 限制Browsing Time在合理范围内（0-60分钟）
df['Browsing Time (min)'] = df['Browsing Time (min)'].clip(0, 60)

# 确保Discount在0-1之间
df['Discount'] = df['Discount'].clip(0, 1)

# 确保Quantity在1-10之间（根据观察）
df['Quantity'] = df['Quantity'].clip(1, 10)

# 确保Age在18-100之间
df['Age'] = df['Age'].clip(18, 100)

数值范围验证:
Quantity范围: 1.0 - 5.0
Discount范围: 0.0 - 0.05
Sales范围: 33.0 - 250.0
Profit范围: 0.5 - 167.5
Age范围: 18 - 70
Browsing Time范围: 1.0 - 40.0


In [7]:
# 创建月份和年份特征
df['Order Month'] = df['Order Date'].dt.month
df['Order Year'] = df['Order Date'].dt.year
df['Order Quarter'] = df['Order Date'].dt.quarter

# 创建折扣金额
df['Discount Amount'] = df['Sales'] * df['Discount']

# 创建净销售额
df['Net Sales'] = df['Sales'] - df['Discount Amount']

# 创建总成本
df['Total Cost'] = df['Shipping Cost'] + (df['Sales'] - df['Profit'])

# 创建利润率
df['Profit Margin'] = df['Profit'] / df['Sales']
df['Profit Margin'] = df['Profit Margin'].replace([np.inf, -np.inf], np.nan)
df['Profit Margin'] = df['Profit Margin'].fillna(0)

# 创建年龄分组
bins = [18, 25, 35, 50, 65, 100]
labels = ['18-24', '25-34', '35-49', '50-64', '65+']
df['Age Group'] = pd.cut(df['Age'], bins=bins, labels=labels, right=False)

# 创建互动总分
df['Engagement Score'] = df['Like'] + df['Share'] + df['Add to Cart']

In [8]:
# 最终数据检查
print("清洗后数据形状:", df.shape)
print("\n清洗后数据类型:")
print(df.dtypes)
print("\n清洗后缺失值统计:")
print(df.isnull().sum().sum())

# 查看清洗后的统计信息
print("\n数值列统计信息:")
numeric_cols = ['Sales', 'Quantity', 'Discount', 'Profit', 'Shipping Cost', 
                'Browsing Time (min)', 'Net Sales', 'Profit Margin']
print(df[numeric_cols].describe())

# 查看分类变量的唯一值
print("\n分类变量唯一值数量:")
categorical_cols = ['Segment', 'Region', 'Gender', 'Education', 'Marital Status',
                    'Product Category', 'Product', 'Order Priority', 'Age Group']
for col in categorical_cols:
    print(f"{col}: {df[col].nunique()} 个唯一值")

清洗后数据形状: (51289, 36)

清洗后数据类型:
Customer ID                    object
Customer Name                  object
Segment                        object
City                           object
State                          object
Country                        object
Region                         object
Gender                         object
Age                             int64
Education                      object
Marital Status                 object
Order ID                       object
Order Date             datetime64[ns]
Months                         object
Ship Mode                      object
Product Category               object
Product                        object
Sales                         float64
Quantity                      float64
Discount                      float64
Profit                        float64
Shipping Cost                 float64
Order Priority                 object
Browsing Time (min)           float64
Like                            int64
Share              

In [9]:
# 保存清洗后的数据
df.to_csv('E_commerce_cleaned.csv', index=False)
print("\n清洗后的数据已保存为 'E_commerce_cleaned.csv'")


清洗后的数据已保存为 'E_commerce_cleaned.csv'


In [10]:
# 生成数据质量报告
def generate_data_quality_report(df):
    report = {
        'Total Records': len(df),
        'Total Columns': len(df.columns),
        'Missing Values': df.isnull().sum().sum(),
        'Duplicate Records': df.duplicated().sum(),
        'Data Types': df.dtypes.to_dict(),
        'Numeric Summary': df.describe().to_dict(),
        'Categorical Summary': {}
    }
    
    for col in categorical_cols:
        if col in df.columns:
            report['Categorical Summary'][col] = df[col].value_counts().to_dict()
    
    return report

quality_report = generate_data_quality_report(df)
print("\n数据质量报告摘要:")
print(f"总记录数: {quality_report['Total Records']}")
print(f"总列数: {quality_report['Total Columns']}")
print(f"缺失值总数: {quality_report['Missing Values']}")
print(f"重复记录数: {quality_report['Duplicate Records']}")


数据质量报告摘要:
总记录数: 51289
总列数: 36
缺失值总数: 80
重复记录数: 0


## RFMB-C 12 核心特征（开题落地）

本节从明细数据 `E_commerce.csv`（订单+浏览行为）按客户聚合，计算 **RFM-BC 12 个核心特征**，并在特征计算后统一做 **Min-Max 标准化到 [0,1]**。

### 12 个特征定义（按开题）

- **R（Recency）**
  - 最近一次消费时间间隔：`R_Recency_Days`
  - 平均消费间隔：`R_Avg_Interval_Days`
- **F（Frequency）**
  - 近 30 天消费次数：`F_Orders_30d`
  - 近 90 天消费次数：`F_Orders_90d`
- **M（Monetary）**
  - 近 30 天消费金额：`M_Sales_30d`
  - 客单价：`M_AOV_30d`（若近30天无订单则回退整体客单价）
  - 消费金额方差：`M_Sales_Var_90d`（近90天订单金额方差）
- **B（Browsing）**
  - 近 30 天浏览次数：`B_Browse_Count_30d`
  - 平均浏览时长：`B_Avg_Browse_Time_30d`
  - 浏览深度（页面跳转次数）：`B_Depth_30d`
- **C（Category preference）**
  - 偏好品类 TOP1 占比：`C_Top1_Ratio`
  - 偏好品类 TOP3 覆盖度：`C_Top3_Coverage`

### 重要说明（数据可得性）

- 本数据集没有显式的“页面跳转次数”字段，因此这里将 **近 30 天浏览深度** 近似定义为：近 30 天内浏览/购买的 **不同商品数（unique Product）**，用于刻画浏览的“广度/跳转强度”。
- 对于 R 类特征（天数越小越好），标准化后会做反向处理，保证 **值越大代表越好/越活跃**，便于聚类解释。


In [ ]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans

# ---------- 1) 读取明细数据并做必要清洗 ----------
raw = pd.read_csv('E_commerce.csv')

# 字段统一与类型转换
raw['Order Date'] = pd.to_datetime(raw['Order Date'], errors='coerce')

# Sales 可能是带 $/逗号的字符串
s = raw['Sales'].astype(str).str.strip()
s = s.str.replace(r'^\((.*)\)$', r'-\1', regex=True)
s = s.str.replace(r'[\$,]', '', regex=True)
raw['Sales'] = pd.to_numeric(s.replace('', np.nan), errors='coerce')

# Browsing Time
raw['Browsing Time (min)'] = pd.to_numeric(raw['Browsing Time (min)'], errors='coerce')

# 基础缺失值填充（只对本节用到的列）
raw['Sales'] = raw['Sales'].fillna(raw['Sales'].median())
raw['Browsing Time (min)'] = raw['Browsing Time (min)'].fillna(raw['Browsing Time (min)'].median())
raw['Order Date'] = raw['Order Date'].fillna(raw['Order Date'].mode(dropna=True)[0])

# 关键维度列
ID_COL = 'Customer ID'
NAME_COL = 'Customer Name'
DATE_COL = 'Order Date'
CAT_COL = 'Product Category'
PROD_COL = 'Product'

# 作为时间窗口的“参考日期”：取全数据最大日期（也可以改为每个客户自己的最后日期，这里用全局便于统一）
ref_date = raw[DATE_COL].max()

# ---------- 2) 为每个客户计算 R / F / M / B / C 特征 ----------

g = raw.sort_values([ID_COL, DATE_COL]).copy()

# (R1) 最近一次消费间隔：ref_date - last_order_date
last_date = g.groupby(ID_COL)[DATE_COL].max()
R_Recency_Days = (ref_date - last_date).dt.days.astype(float)

# (R2) 平均消费间隔：同一客户订单日期的 diff 均值
# 先算每个客户的相邻订单间隔
intervals = g.groupby(ID_COL)[DATE_COL].diff().dt.days
R_Avg_Interval_Days = intervals.groupby(g[ID_COL]).mean().astype(float)

# 时间窗口过滤
mask_30 = g[DATE_COL] >= (ref_date - pd.Timedelta(days=30))
mask_90 = g[DATE_COL] >= (ref_date - pd.Timedelta(days=90))

# (F) 近 30/90 天订单数（以订单行计数）
F_Orders_30d = g[mask_30].groupby(ID_COL).size().astype(float)
F_Orders_90d = g[mask_90].groupby(ID_COL).size().astype(float)

# (M1) 近 30 天消费金额（Sales 求和）
M_Sales_30d = g[mask_30].groupby(ID_COL)['Sales'].sum().astype(float)

# (M2) 客单价：近 30 天 AOV（若 30 天无订单则回退整体 AOV）
AOV_all = g.groupby(ID_COL)['Sales'].mean().astype(float)
AOV_30d = g[mask_30].groupby(ID_COL)['Sales'].mean().astype(float)
M_AOV_30d = AOV_30d.reindex(AOV_all.index).fillna(AOV_all)

# (M3) 近 90 天消费金额方差
M_Sales_Var_90d = g[mask_90].groupby(ID_COL)['Sales'].var(ddof=0).astype(float)

# (B1) 近 30 天浏览次数：用近 30 天记录条数近似（每条记录对应一次浏览/购买行为）
B_Browse_Count_30d = g[mask_30].groupby(ID_COL).size().astype(float)

# (B2) 近 30 天平均浏览时长
B_Avg_Browse_Time_30d = g[mask_30].groupby(ID_COL)['Browsing Time (min)'].mean().astype(float)

# (B3) 近 30 天浏览深度：近 30 天不同商品数（unique Product）近似“页面跳转次数”
B_Depth_30d = g[mask_30].groupby(ID_COL)[PROD_COL].nunique().astype(float)

# (C1/C2) 偏好品类占比：以全周期为准（也可改为近90天）
cat_counts = g.groupby([ID_COL, CAT_COL]).size().rename('cnt').reset_index()
cat_totals = cat_counts.groupby(ID_COL)['cnt'].sum()
cat_sorted = cat_counts.sort_values([ID_COL, 'cnt'], ascending=[True, False])

# Top1
cat_top1 = cat_sorted.groupby(ID_COL).head(1).set_index(ID_COL)['cnt']
C_Top1_Ratio = (cat_top1 / cat_totals).astype(float)

# Top3 coverage
top3_sum = cat_sorted.groupby(ID_COL).head(3).groupby(ID_COL)['cnt'].sum()
C_Top3_Coverage = (top3_sum / cat_totals).astype(float)

# 把所有特征合并到一个 DataFrame
features = pd.DataFrame({
    'R_Recency_Days': R_Recency_Days,
    'R_Avg_Interval_Days': R_Avg_Interval_Days,
    'F_Orders_30d': F_Orders_30d,
    'F_Orders_90d': F_Orders_90d,
    'M_Sales_30d': M_Sales_30d,
    'M_AOV_30d': M_AOV_30d,
    'M_Sales_Var_90d': M_Sales_Var_90d,
    'B_Browse_Count_30d': B_Browse_Count_30d,
    'B_Avg_Browse_Time_30d': B_Avg_Browse_Time_30d,
    'B_Depth_30d': B_Depth_30d,
    'C_Top1_Ratio': C_Top1_Ratio,
    'C_Top3_Coverage': C_Top3_Coverage,
}).reset_index().rename(columns={ID_COL: 'Customer_ID'})

# 补齐姓名（取该客户最常见姓名）
name_map = g.groupby(ID_COL)[NAME_COL].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else x.iloc[0])
features['Customer_Name'] = features['Customer_ID'].map(name_map).fillna('Unknown')

# 缺失值处理：方差/间隔在订单很少时可能缺失
for c in [
    'R_Avg_Interval_Days',
    'F_Orders_30d',
    'F_Orders_90d',
    'M_Sales_30d',
    'M_Sales_Var_90d',
    'B_Browse_Count_30d',
    'B_Avg_Browse_Time_30d',
    'B_Depth_30d',
    'C_Top1_Ratio',
    'C_Top3_Coverage',
]:
    if c in features.columns:
        features[c] = features[c].fillna(0.0)

# ---------- 3) Min-Max 标准化到 [0,1]（R 类反向） ----------
# 为保证 notebook 与项目默认数据源逻辑完全一致，这里直接调用脚本生成 RFMB-C 特征数据。
# 输出文件：customer_features_rfmbc.csv（画像字段 + 12 个 RFMB-C 特征（Min-Max）+ Cluster 标签）

from build_rfmbc_features import main

main()

# 预览输出
import pandas as pd

df_rfmbc = pd.read_csv('customer_features_rfmbc.csv')
print('customer_features_rfmbc.csv shape=', df_rfmbc.shape)
df_rfmbc.head()